In [3]:
# ==========================================================
# INSURANCE COST PREDICTION USING ONE MULTI-LAYER NEURAL NETWORK
# Google Colab | NumPy | Pandas | TensorFlow/Keras
# ==========================================================

import os
import zipfile
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
tf.random.set_seed(42)

# 1. UPLOAD DATASET
print("Upload insurance.csv or a ZIP file containing insurance.csv")
uploaded = files.upload()

if not uploaded:
    raise ValueError("Please upload your dataset.")

uploaded_name = next(iter(uploaded))
data_dir = "/content/insurance_data"
os.makedirs(data_dir, exist_ok=True)

if uploaded_name.lower().endswith(".zip"):
    with zipfile.ZipFile(uploaded_name, "r") as archive:
        archive.extractall(data_dir)

    csv_paths = [
        os.path.join(root, name)
        for root, _, names in os.walk(data_dir)
        for name in names
        if name.lower() == "insurance.csv"
    ]

    if not csv_paths:
        raise FileNotFoundError("insurance.csv was not found in the ZIP.")

    csv_path = csv_paths[0]

elif uploaded_name.lower().endswith(".csv"):
    csv_path = os.path.join(data_dir, "insurance.csv")
    shutil.copy(uploaded_name, csv_path)

else:
    raise ValueError("Upload a .csv or .zip file.")

# 2. LOAD AND CLEAN DATA
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()

required = [
    "age", "sex", "bmi", "children",
    "smoker", "region", "charges"
]

missing_columns = set(required) - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

df = df[required].copy()

for col in ["age", "bmi", "children", "charges"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["sex", "smoker", "region"]:
    df[col] = df[col].astype("string").str.strip().str.lower()

df = df.dropna().drop_duplicates().reset_index(drop=True)

if df.empty:
    raise ValueError("No valid rows remain after cleaning.")

if (df["charges"] < 0).any():
    raise ValueError("Insurance charges cannot be negative.")

print("\nCleaned dataset shape:", df.shape)
display(df.head())
print("\nSummary statistics:")
display(df.describe())

df.to_csv("/content/insurance_cleaned.csv", index=False)

# 3. NUMPY AND PANDAS ANALYSIS
charges = df["charges"].to_numpy(dtype=np.float64)

print("\nMean charges:", np.mean(charges))
print("Median charges:", np.median(charges))
print("Minimum charges:", np.min(charges))
print("Maximum charges:", np.max(charges))
print("Standard deviation:", np.std(charges))

print("\nAverage charges by smoker status:")
display(df.groupby("smoker")["charges"].mean())

# 4. PREPARE FEATURES AND TARGET
X_df = df.drop(columns=["charges"])
y = df["charges"].to_numpy(dtype=np.float32).reshape(-1, 1)

# Convert categorical features to numeric columns
X_df = pd.get_dummies(
    X_df,
    columns=["sex", "smoker", "region"],
    dtype=float
)

feature_names = X_df.columns.tolist()
X = X_df.to_numpy(dtype=np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 5. SCALE FEATURES AND TARGET
x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = x_scaler.transform(X_test).astype(np.float32)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train).astype(np.float32)

print("\nTraining examples:", len(X_train))
print("Testing examples:", len(X_test))
print("Number of input features:", X_train.shape[1])

# 6. BUILD ONE MULTI-LAYER NEURAL NETWORK
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1)
], name="Insurance_Multi_Layer_Neural_Network")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

print("\nNeural network architecture:")
model.summary()

# 7. TRAIN THE SINGLE MODEL
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=25,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_split=0.20,
    epochs=300,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# 8. EVALUATE MODEL ON TEST DATA
pred_scaled = model.predict(X_test_scaled, verbose=0)
predictions = y_scaler.inverse_transform(pred_scaled).flatten()
actual = y_test.flatten()

mae = mean_absolute_error(actual, predictions)
rmse = np.sqrt(mean_squared_error(actual, predictions))
r2 = r2_score(actual, predictions)

print("\n========== MODEL PERFORMANCE ==========")
print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")

results = pd.DataFrame([{
    "Model": "Insurance_Multi_Layer_Neural_Network",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2,
    "Epochs_Trained": len(history.history["loss"])
}])

display(results)
results.to_csv("/content/neural_network_results.csv", index=False)

# 9. SAVE ACTUAL AND PREDICTED CHARGES
predictions_df = pd.DataFrame({
    "Actual_Charges": actual,
    "Predicted_Charges": predictions,
    "Absolute_Error": np.abs(actual - predictions)
})

print("\nSample predictions:")
display(predictions_df.head(10))

predictions_df.to_csv(
    "/content/insurance_test_predictions.csv",
    index=False
)

# 10. SAVE THE MODEL AND PREPROCESSING INFORMATION
model_path = "/content/insurance_neural_network.keras"
model.save(model_path)

np.savez(
    "/content/insurance_preprocessing.npz",
    x_mean=x_scaler.mean_,
    x_scale=x_scaler.scale_,
    y_mean=y_scaler.mean_,
    y_scale=y_scaler.scale_,
    feature_names=np.array(feature_names, dtype=str)
)

# Save feature names separately for convenience
pd.DataFrame({"feature_name": feature_names}).to_csv(
    "/content/insurance_feature_names.csv",
    index=False
)

# 11. SAVE TRAINING HISTORY
pd.DataFrame(history.history).to_csv(
    "/content/training_history.csv",
    index=False
)

# 12. PACKAGE PROJECT FILES
project_dir = "/content/insurance_neural_network_project"
os.makedirs(project_dir, exist_ok=True)

output_files = [
    "/content/insurance_cleaned.csv",
    "/content/neural_network_results.csv",
    "/content/insurance_test_predictions.csv",
    "/content/insurance_preprocessing.npz",
    "/content/insurance_feature_names.csv",
    "/content/training_history.csv",
    "/content/insurance_neural_network.keras"
]

for path in output_files:
    shutil.copy(path, project_dir)

with open(os.path.join(project_dir, "README.txt"), "w") as f:
    f.write(
        "Insurance Cost Prediction using one multi-layer neural network.\n"
        "Libraries: NumPy, Pandas, TensorFlow/Keras, scikit-learn.\n"
        "Model: Dense(64) -> Dense(32) -> Dense(16) -> Dense(1).\n"
        "See neural_network_results.csv for test performance.\n"
    )

zip_path = shutil.make_archive(
    "/content/insurance_neural_network_project",
    "zip",
    project_dir
)

print("\nProject completed!")
print("Model saved:", model_path)
print("Results saved:", "/content/neural_network_results.csv")
print("Project package:", zip_path)

# 13. DOWNLOAD PROJECT ZIP
files.download(zip_path)

Upload insurance.csv or a ZIP file containing insurance.csv


Saving archive.zip to archive (2).zip

Cleaned dataset shape: (1337, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520



Summary statistics:


,age,bmi,children,charges
count,1337.000000,1337.000000,1337.000000,1337.000000
mean,39.222139,30.663452,1.095737,13279.121487
std,14.044333,6.100468,1.205571,12110.359656
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.290000,0.000000,4746.344000
50%,39.000000,30.400000,1.000000,9386.161300
75%,51.000000,34.700000,2.000000,16657.717450
max,64.000000,53.130000,5.000000,63770.428010


...


Sample predictions:


,Actual_Charges,Predicted_Charges,Absolute_Error
0,8688.858398,8615.497070,73.361328
1,5708.867188,6004.865723,295.998535
2,11436.738281,13438.907227,2002.168945
3,38746.355469,37279.644531,1466.710938
4,4463.205078,7057.697266,2594.492188
5,9304.702148,9390.645508,85.943359
6,38511.628906,37992.910156,518.718750
7,2150.468994,3713.200439,1562.731445
8,7345.726562,7321.788574,23.937988
9,10264.442383,10709.307617,444.865234



Project completed!
Model saved: /content/insurance_neural_network.keras
Results saved: /content/neural_network_results.csv
Project package: /content/insurance_neural_network_project.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>